# 10 — Replay → token batch → masked PPO mechanics

Тетрадка объясняет, какие именно tensors приходят в token-level learning. Она использует replay из notebook 09 и не вызывает LLM.


In [ ]:
from pathlib import Path
import json
import jax.numpy as jnp
from tunix_craftext.research.algorithms import masked_token_ppo_loss, masked_token_returns
from tunix_craftext.replay import ReplayArtifact, ReplayStep
from tunix_craftext.text_trajectory import text_trajectory_from_replay


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
path = ROOT / 'artifacts/trajectories/batched-qwen-rollout/env-0.json'
if not path.is_file(): raise FileNotFoundError('Run notebook 09 first: ' + str(path))
payload = json.loads(path.read_text())
steps = tuple(ReplayStep(**step) for step in payload['steps'])
replay = ReplayArtifact(payload['config_path'], payload['commit'], payload['backend'], steps, payload['schema'])
batch = text_trajectory_from_replay(replay)
print('token ids:', batch.token_ids.shape, 'prompt ids:', batch.prompt_token_ids.shape)
print('token mask:', batch.token_mask.tolist())
print('policy mask:', batch.policy_mask.tolist(), '(fallback rows are excluded)')


In [ ]:
returns = masked_token_returns(batch.rewards, batch.token_mask, gamma=0.99)
advantages = returns  # A real learner subtracts critic values here.
zeros = jnp.zeros_like(batch.old_logprobs)
loss, metrics = masked_token_ppo_loss(
    batch.old_logprobs, batch.old_logprobs, advantages, zeros, zeros, returns, batch.policy_mask,
    clip_epsilon=0.2, value_coefficient=0.5, entropy=zeros, entropy_coefficient=0.01,
)
print('returns:', returns.tolist())
print('loss:', float(loss))
print({name: float(value) for name, value in metrics.items()})


`old_logprobs` использованы и как `new_logprobs` только для иллюстрации формы loss. Реальное обучение должно пересчитать logprobs trainable actor, получить trainable critic values и сделать Optax update; именно это следующий integration slice.
